In [0]:
# ============================================
# Databricks Notebook
# S3 -> Unity Catalog Delta Table
# ============================================

# Notebook parameters
dbutils.widgets.text("s3_path", "")
dbutils.widgets.text("target_table", "")

s3_path = dbutils.widgets.get("s3_path")
target_table = dbutils.widgets.get("target_table")

print(f"S3 path      : {s3_path}")
print(f"Target table : {target_table}")

if not s3_path:
    raise ValueError("s3_path parameter is required")

if not target_table:
    raise ValueError("target_table parameter is required")


# ============================================
# Read CSV from S3
# ============================================

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(s3_path)
)

print(f"Record count: {df.count()}")

df.printSchema()
df.show(10, truncate=False)


# ============================================
# Write to Unity Catalog
# ============================================

(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Successfully loaded data into {target_table}")